# 데이터셋 규모 표 — 세션 / 윈도우 / split (2026-07-27)

원본 parquet 과 전처리 산출물(`preprocessed_MM_<method>/`)에서 **운동별 세션 수 →
윈도우 수 → split 별 분배 → test set 잔량**을 순서대로 찍는다.

**정의**
- **세션** = 한 사람이 한 운동을 한 세트 수행한 녹화 1건. 라벨은 세션당 1종이고,
  split 은 세션 단위로 잘린다(`data_preprocess_MM.py` 의 `split_sessions`).
- **윈도우** = 5초 window / 0.5초 stride → **인접 윈도우끼리 90% 겹친다**.
  따라서 윈도우 수는 표본 수가 아니다. 실질 독립 단위는 세션이다(§5 참고).

윈도우 수는 회전행렬 R 과 무관하므로 어떤 method 폴더를 읽어도 같아야 한다.
§6 에서 그 일관성을 실제로 검증한다.

In [1]:
from pathlib import Path
import json
import collections

import numpy as np
import pandas as pd

# notebook/ 에서 실행하든 repo root 에서 실행하든 위로 올라가며 repo root 를 찾는다
ROOT = Path.cwd()
while not (ROOT / "data_preprocess_MM.py").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data_preprocess_MM.py").is_file(), f"repo root 를 못 찾음 (cwd={Path.cwd()})"

# 기준 전처리 폴더. 윈도우 수는 method 와 무관하므로 아무거나 하나면 된다.
PREP = ROOT / "preprocessed_MM_pca"

SPLITS = ("train", "val", "test")
PREFIX = {"source": ("train", "val", "test"),
          "target": ("target_train", "target_val", "target_test")}

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

manifest = json.loads((PREP / "split_manifest.json").read_text(encoding="utf-8"))

print("repo root :", ROOT)
print("prep dir  :", PREP.relative_to(ROOT))
print("split     :", manifest["split_ratio"], " seed:", manifest["split_seed"])

repo root : /home/user1/Domain_Adaptation_for_EMG_IMU_Sensor
prep dir  : preprocessed_MM_pca
split     : [0.7, 0.15, 0.15]  seed: 42


## 0. 로더

`split_manifest.json` 에 도메인별 parquet 경로와 세션 컬럼명이 적혀 있으므로,
경로를 노트북에 다시 하드코딩하지 않고 manifest 에서 읽는다(= provenance 일치 보장).

In [2]:
def session_labels(domain):
    """도메인의 '세션명 -> 운동' Series. parquet 경로/세션컬럼은 manifest 에서 읽는다."""
    info = manifest[domain]
    df = pd.read_parquet(ROOT / info["parquet"], columns=[info["session_col"], "exercise"])
    return (df.drop_duplicates(info["session_col"])
              .set_index(info["session_col"])["exercise"])


def window_labels(domain):
    """도메인의 split -> 윈도우 라벨 배열 dict."""
    return {s: np.load(PREP / f"y_{p}.npy", allow_pickle=True)
            for s, p in zip(SPLITS, PREFIX[domain])}


SESS = {d: session_labels(d) for d in ("source", "target")}          # 세션명 -> 운동
WIN  = {d: window_labels(d)  for d in ("source", "target")}          # split -> (N,) 라벨
SPLIT_SESS = {d: {s: SESS[d].reindex(manifest[d][s]) for s in SPLITS}
              for d in ("source", "target")}                          # split 별 세션 라벨

CLASSES = sorted(SESS["source"].unique())
assert CLASSES == sorted(SESS["target"].unique()), "두 도메인의 운동 종류가 다름"
print(f"운동 {len(CLASSES)}종:", ", ".join(CLASSES))

운동 10종: barbellcurl, barbellrow, benchpress, bte, deadlift, dips, latpulldown, ohp, pullup, pushup


## 1. 운동별 세션 수

세션 = 한 사람의 한 세트. 이게 이 데이터셋의 **실질 표본 단위**다.

In [3]:
tbl1 = pd.DataFrame({d: SESS[d].value_counts().reindex(CLASSES) for d in ("source", "target")})
tbl1.index.name = "exercise"
tbl1.loc["합계"] = tbl1.sum()
display(tbl1)

_c = tbl1.drop(index="합계")
for d in ("source", "target"):
    print(f"{d}: 세션 {int(_c[d].sum())}, 클래스별 min={int(_c[d].min())}({_c[d].idxmin()}) "
          f"max={int(_c[d].max())}({_c[d].idxmax()}) → 불균형 {_c[d].max()/_c[d].min():.2f}x")

,source,target
exercise,,
barbellcurl,44,51
barbellrow,61,52
benchpress,60,54
bte,42,53
deadlift,64,42
dips,47,41
latpulldown,59,56
ohp,60,48
pullup,34,48


source: 세션 508, 클래스별 min=34(pullup) max=64(deadlift) → 불균형 1.88x
target: 세션 484, 클래스별 min=39(pushup) max=56(latpulldown) → 불균형 1.44x


## 2. 윈도우로 자른 후

5초 window / 0.5초 stride. 세션 길이가 제각각이라 세션당 윈도우 수도 도메인별로 다르다.

In [4]:
rows = []
for d in ("source", "target"):
    n_sess = len(SESS[d])
    n_win = sum(len(v) for v in WIN[d].values())
    rows.append({"domain": d, "세션": n_sess, "윈도우": n_win,
                 "세션당 평균 윈도우": round(n_win / n_sess, 1),
                 "세션당 평균 길이(초)": round((n_win / n_sess - 1) * 0.5 + 5.0, 1)})
tbl2 = pd.DataFrame(rows).set_index("domain")
display(tbl2)

,세션,윈도우,세션당 평균 윈도우,세션당 평균 길이(초)
domain,,,,
source,508,11706,23.0,16.0
target,484,21629,44.7,26.8


## 3. split 별 세션 / 윈도우

세션 비율은 정확히 70/15/15 로 맞지만, 세션 길이가 균일하지 않아 **윈도우 비율은 조금 어긋난다**.

In [5]:
rows = []
for d in ("source", "target"):
    tot_w = sum(len(v) for v in WIN[d].values())
    for s in SPLITS:
        n_sess, n_win = len(manifest[d][s]), len(WIN[d][s])
        rows.append({"domain": d, "split": s, "세션": n_sess,
                     "세션%": f"{n_sess/len(SESS[d]):.1%}",
                     "윈도우": n_win, "윈도우%": f"{n_win/tot_w:.1%}"})
tbl3 = pd.DataFrame(rows).set_index(["domain", "split"])
display(tbl3)

세션    세션%    윈도우   윈도우%
domain split                          
source train  354  69.7%   8027  68.6%
       val     77  15.2%   1795  15.3%
       test    77  15.2%   1884  16.1%
target train  338  69.8%  15134  70.0%
       val     73  15.1%   3089  14.3%
       test    73  15.1%   3406  15.7%

## 4. TEST SET 에 남는 양 (최종 보고용)

`test` 는 model selection 에 쓰지 않고 최종 보고에만 쓰는 split 이다
(`protocol-train-val-test-split`). 클래스별로 얼마나 남는지가 신뢰구간을 좌우한다.

In [6]:
def per_class(domain, split):
    sess = SPLIT_SESS[domain][split].value_counts().reindex(CLASSES).fillna(0).astype(int)
    win = pd.Series(collections.Counter(WIN[domain][split].tolist())).reindex(CLASSES).fillna(0).astype(int)
    return sess, win

tbl4 = pd.DataFrame({
    (d, k): v
    for d in ("source", "target")
    for k, v in zip(("세션", "윈도우"), per_class(d, "test"))
})
tbl4.index.name = "exercise"
tbl4.loc["합계"] = tbl4.sum()
display(tbl4)

source       target      
                세션   윈도우     세션   윈도우
exercise                             
barbellcurl      7   195      8   408
barbellrow       9   180      8   363
benchpress       9   192      8   334
bte              6   129      8   332
deadlift        10   289      6   375
dips             7   155      6   269
latpulldown      9   226      9   508
ohp              9   265      7   351
pullup           5   105      7   203
pushup           6   148      6   263
합계              77  1884     73  3406

## 5. 이 숫자를 어떻게 읽어야 하는가

윈도우 수를 표본 수로 읽으면 안 된다. 90% 오버랩이라 인접 윈도우는 4.5초를 공유한다.
겹치지 않는 5초 구간으로 환산하면 실질 독립 표본은 대략 **1/10** 이고, 세션 단위로 분할하므로
진짜 독립 단위는 **세션 수**다. 아래 표의 `최소 클래스 세션수` 가 곧 그 클래스 정확도의 해상도다.

In [7]:
rows = []
for d in ("source", "target"):
    sess, win = per_class(d, "test")
    eff = win.sum() / 10          # 비중첩 5초 구간 환산 (stride 0.5s / window 5s)
    rows.append({
        "domain": d,
        "test 세션": int(sess.sum()),
        "test 윈도우": int(win.sum()),
        "비중첩 환산 ≈": int(round(eff)),
        "클래스별 윈도우 min": int(win.min()),
        "클래스별 윈도우 max": int(win.max()),
        "윈도우 불균형": f"{win.max()/win.min():.2f}x",
        # 동점 클래스가 여럿일 수 있으므로 idxmin 하나만 쓰지 않는다
        "최소 클래스 세션수": f"{int(sess.min())} ({', '.join(sess[sess == sess.min()].index)})",
        "세션 1개당 해당 클래스 정확도 변동": f"{1/sess.min():.1%}p",
    })
display(pd.DataFrame(rows).set_index("domain").T)

print("→ 클래스 불균형이 2배를 넘으므로 overall accuracy 는 다수 클래스에 가중된다.")
print("  macro-F1 / balanced accuracy 를 함께 보고하는 편이 안전하다.")

domain,source,target
test 세션,77,73
test 윈도우,1884,3406
비중첩 환산 ≈,188,341
클래스별 윈도우 min,105,203
클래스별 윈도우 max,289,508
윈도우 불균형,2.75x,2.50x
최소 클래스 세션수,5 (pullup),"6 (deadlift, dips, pushup)"
세션 1개당 해당 클래스 정확도 변동,20.0%p,16.7%p


→ 클래스 불균형이 2배를 넘으므로 overall accuracy 는 다수 클래스에 가중된다.
  macro-F1 / balanced accuracy 를 함께 보고하는 편이 안전하다.


## 6. method 폴더 간 윈도우 수 일관성 검증

R 은 IMU 축을 회전시킬 뿐 윈도우 개수를 바꾸지 않는다. 따라서 모든 `preprocessed_MM_*`
폴더의 6개 split 크기는 **완전히 같아야 한다**. 다르면 서로 다른 분할로 전처리된 것이므로
method 간 비교가 성립하지 않는다.

In [8]:
rows = []
for d in sorted(ROOT.glob("preprocessed_MM_*")):
    if not d.is_dir():
        continue
    try:
        counts = {p: len(np.load(d / f"y_{p}.npy", allow_pickle=True))
                  for p in PREFIX["source"] + PREFIX["target"]}
    except FileNotFoundError as e:
        rows.append({"dir": d.name, "상태": f"불완전 ({Path(e.filename).name} 없음)"})
        continue
    same = json.loads((d / "split_manifest.json").read_text(encoding="utf-8"))
    rows.append({"dir": d.name, "상태": "ok",
                 **counts,
                 "seed": same["split_seed"],
                 "동일분할": same["source"]["test"] == manifest["source"]["test"]})
tbl6 = pd.DataFrame(rows).set_index("dir")
# 불완전 폴더의 NaN 때문에 카운트 컬럼이 float 으로 승격되는 것을 막는다
_num = [c for c in tbl6.columns if c in PREFIX["source"] + PREFIX["target"] or c == "seed"]
tbl6[_num] = tbl6[_num].astype("Int64")
display(tbl6)

ok = tbl6[tbl6["상태"] == "ok"]
sizes = ok[list(PREFIX["source"] + PREFIX["target"])].drop_duplicates()
print("윈도우 수 조합 종류:", len(sizes), "(1 이어야 정상)")
print("split_manifest 세션까지 동일:", bool(ok["동일분할"].all()))

,상태,train,val,test,target_train,target_val,target_test,seed,동일분할
dir,,,,,,,,,
preprocessed_MM_gravity,ok,8027,1795,1884,15134,3089,3406,42,True
preprocessed_MM_kabsch,ok,8027,1795,1884,15134,3089,3406,42,True
preprocessed_MM_learned,불완전 (y_test.npy 없음),<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN
preprocessed_MM_pca,ok,8027,1795,1884,15134,3089,3406,42,True
preprocessed_MM_permutation,ok,8027,1795,1884,15134,3089,3406,42,True
preprocessed_MM_raw,ok,8027,1795,1884,15134,3089,3406,42,True
preprocessed_MM_raw_isotropic,ok,8027,1795,1884,15134,3089,3406,42,True


윈도우 수 조합 종류: 1 (1 이어야 정상)
split_manifest 세션까지 동일: True
